# Local thickness: how thick is it, everywhere?

**Local thickness** answers a deceptively simple question: *how thick is this structure at every point?* It is used for the walls of shells and foams, trabecular bone, pore and fibre diameters, and anywhere else where "size" is not one number but varies through the object.

**The idea.** For every foreground voxel, find the largest sphere that fits entirely inside the object *and contains that voxel*. The radius of that sphere is the local thickness of the voxel. The result is a **thickness map**: a volume of the same shape as the input, with 0 in the background.

qim3d wraps the *Fast Local Thickness* algorithm (Dahl & Dahl, CVPRW 2023), which makes this practical for large 3D scans.

In this notebook you will:

1. build intuition on a tiny 2D shape where we know the right answer,
2. measure the wall of a real 3D shell: segment, compute, visualize,
3. extract numbers from the thickness map,
4. see which choices change the result (threshold, `scale`, `mask`).

In [ ]:
import numpy as np
import qim3d

## Part 1: Intuition on a shape where we know the answer

Two discs of radius 30 and 20 joined by a narrow bridge of height 12 (radius 6). We build it with plain NumPy, so we know the correct thickness everywhere.

In [ ]:
yy, xx = np.mgrid[0:120, 0:230]

big_disc   = (yy - 60) ** 2 + (xx - 50) ** 2  <= 30 ** 2    # radius 30
small_disc = (yy - 60) ** 2 + (xx - 175) ** 2 <= 20 ** 2    # radius 20
bridge     = (yy >= 54) & (yy < 66) & (xx > 50) & (xx < 175)  # 12 voxels high -> radius 6

dumbbell = big_disc | small_disc | bridge   # a binary (boolean) image

In [ ]:
dumbbell

`qim3d.processing.local_thickness` works on 2D and 3D images. With `visualize=True` it also shows the input, the thickness map and a histogram of the thickness values:

In [ ]:
lt_dumbbell = qim3d.processing.local_thickness(dumbbell, visualize=True)

In [ ]:
print(f"centre of big disc   (true radius 30): {lt_dumbbell[60, 50]:.2f}")
print(f"centre of small disc (true radius 20): {lt_dumbbell[60, 175]:.2f}")
print(f"middle of the bridge (true radius  6): {lt_dumbbell[60, 112]:.2f}")

Two things to notice:

- **The map is flat inside each part.** Every voxel gets the radius of the largest sphere that *contains* it, not the distance to the nearest edge. The edge of a disc therefore has the same value as its centre.
- **The values are radii, not diameters.** The disc of diameter 60 gives 30. For the *diameter* (the usual "thickness" of a wall), multiply by 2. This matters whenever you report numbers.

Values are in **voxels**. To convert to physical units, multiply by the voxel size.

## Part 2: A real volume, the wall of a shell

`qim3d.examples.shell_225x128x128` is a 3D scan of a shell: a thin wall of varying thickness enclosing several chambers. Our goal is to measure that wall.

In [ ]:
shell = qim3d.examples.shell_225x128x128

print(f"Shape: {shell.shape}, dtype: {shell.dtype}, intensity range: {shell.min()} .. {shell.max()}")
qim3d.viz.slices_grid(shell, n_slices=6, image_size=3)

### Step 1: Segment

Local thickness is defined for a **binary** image, so first we must decide what is "shell" and what is not. If you pass a grayscale volume, qim3d binarizes it with Otsu's method and logs a warning. It is better to do this yourself, because **the threshold is part of the measurement** (we will see how much it matters in Part 4).

The histogram shows the intensities, and `qim3d.operations.threshold` computes a threshold (here Otsu's method), drawn as a vertical line:

In [ ]:
thr = qim3d.operations.threshold(shell, method="otsu")
print(f"Otsu threshold: {thr}")

qim3d.viz.histogram(shell, bins=64, vertical_line=thr);

In [ ]:
binary = shell > thr
print(f"Shell occupies {100 * binary.mean():.1f}% of the volume")

Always check a segmentation visually. Use the slider of the overlay to fade between the data and the binary result:

In [ ]:
qim3d.viz.overlay(shell, binary, colormaps=["gray", "viridis"])

### Step 2: Compute the thickness map

This takes a few seconds on a volume of this size.

In [ ]:
lt = qim3d.processing.local_thickness(binary)

print(f"Thickness map: shape {lt.shape}, dtype {lt.dtype}")
print(f"Largest local thickness (radius): {lt.max():.2f} voxels")

### Step 3: Look at the result

`qim3d.viz.local_thickness` gives an interactive view with the original slice, the thickness map and a histogram of thickness values. Scroll through the slices with the slider (`axis` chooses the slicing direction):

In [ ]:
qim3d.viz.local_thickness(shell, lt, axis=0)

With `max_projection=True` you get one static figure instead, with the maximum along the chosen axis. This is useful for reports:

In [ ]:
qim3d.viz.local_thickness(shell, lt, axis=0, max_projection=True)

For a fixed set of slices, for example for a slide or a paper, use `slices_grid` with a colormap and a colorbar:

In [ ]:
qim3d.viz.slices_grid(lt, n_slices=10, colormap="viridis", colorbar=True, image_size=3)

And in 3D, with the colour showing the thickness. Rotate with the mouse:

In [ ]:
qim3d.viz.volumetric(lt, colormap="viridis")

## Part 3: From a map to numbers

A thickness map is only useful if you can answer questions with it. Everything below is plain NumPy on the map.

### Summary statistics

Only voxels with `lt > 0` belong to the shell. We convert radius to **diameter** so the numbers describe the wall thickness:

In [ ]:
radius = lt[lt > 0]
diameter = 2 * radius    # wall thickness in voxels

print(f"Voxels in the shell: {diameter.size}")
print(f"Wall thickness (diameter, voxels):")
print(f"   mean   {diameter.mean():.2f}  (std {diameter.std():.2f})")
print(f"   median {np.median(diameter):.2f}")
print(f"   5th-95th percentile: {np.percentile(diameter, 5):.2f} - {np.percentile(diameter, 95):.2f}")
print(f"   maximum {diameter.max():.2f}")

The example volume has no voxel-size calibration, so everything here is in voxels. For a real scan, multiply by the voxel size, e.g. `diameter_um = diameter * voxel_size_um`.

### Distribution

The histogram ignores zero by default, so only the shell is counted. We plot the diameter, with the mean and the median marked:

In [ ]:
qim3d.viz.histogram(2 * lt, bins=40, vertical_line=[diameter.mean(), np.median(diameter)]);

### Where is the wall thin?

Weak points are often more interesting than averages. Let's flag the thinnest 10% of the shell and see where they are. We build a small label volume (1 = shell, 2 = thin) and display it with a categorical colormap:

In [ ]:
thin_limit = np.percentile(diameter, 10)
thin = (lt > 0) & (2 * lt <= thin_limit)

print(f"Thinnest 10% of the shell: diameter <= {thin_limit:.2f} voxels")

regions = np.zeros(lt.shape, dtype=np.uint8)
regions[lt > 0] = 1     # shell
regions[thin] = 2       # thin parts

cmap = qim3d.viz.colormaps.segmentation(n_labels=2)
qim3d.viz.slices_grid(regions, n_slices=6, colormap=cmap, max_value=2, image_size=3)

### Thickness along the object

Because the thickness map is just an array, we can slice it. Here we split the shell into three equal parts along axis 0 and compare them:

In [ ]:
n = lt.shape[0]
parts = {"first third": lt[: n // 3], "middle third": lt[n // 3 : 2 * n // 3], "last third": lt[2 * n // 3 :]}

for name, part in parts.items():
    d = 2 * part[part > 0]
    print(f"{name:13s} mean wall thickness {d.mean():.2f} voxels  (n = {d.size})")

## Part 4: What changes the answer

### The threshold

Because we measure the *segmented* object, a different threshold gives a different thickness. Let's compare Otsu's method, Li's method, and a manually chosen higher value:

In [ ]:
candidates = {
    "otsu": qim3d.operations.threshold(shell, method="otsu"),
    "li": qim3d.operations.threshold(shell, method="li"),
    "manual": 100,
}

print(f"{'method':8s} {'threshold':>9s} {'shell %':>8s} {'mean diameter':>14s}")
for name, t in candidates.items():
    lt_t = qim3d.processing.local_thickness(shell > t)
    print(f"{name:8s} {t:9.1f} {100 * (shell > t).mean():8.1f} {2 * lt_t[lt_t > 0].mean():14.2f}")

The differences are substantial. **Choose the threshold deliberately, report it, and keep it fixed when comparing samples.**

### Speed vs accuracy: `scale`

`scale` computes the thickness on a downsampled copy to save time. The result is returned at the original size. This is handy for exploring, but the values are underestimated, so use `scale=1` for final numbers.

In [ ]:
import time

print(f"{'scale':>5s} {'time (s)':>9s} {'mean diameter':>14s}")
for s in [1, 0.5, 0.25]:
    t0 = time.perf_counter()
    lt_s = qim3d.processing.local_thickness(binary, scale=s)
    dt = time.perf_counter() - t0
    print(f"{s:5.2f} {dt:9.2f} {2 * lt_s[lt_s > 0].mean():14.2f}")

### Restricting to a region: `mask`

To measure only a region of interest, pass a boolean `mask`. Thickness is reported only inside the mask and is 0 elsewhere. Here we keep only the central part of the shell:

In [ ]:
roi = np.zeros(binary.shape, dtype=bool)
roi[60:160] = True

lt_roi = qim3d.processing.local_thickness(binary, mask=roi)

inside = roi & binary
print(f"Voxels with thickness outside the mask: {(lt_roi[~roi] > 0).sum()}")
print(f"Mean diameter in the region: {2 * lt_roi[inside].mean():.2f}  (full computation: {2 * lt[inside].mean():.2f})")

Values close to the border of the mask can differ slightly from the unmasked computation.

## Your turn

- Redo the analysis on `qim3d.examples.bone_128x128x128` (trabecular bone). Check the histogram first and choose your threshold.
- Apply `local_thickness` to the *inverted* binary image (`~binary`) to measure the empty space instead. What does the large background around the object do to the result?
- Find the thickest 5% of the shell and display where it is, as we did for the thin parts.

In [ ]:
# Your experiments here

## Reference

Dahl, V. A., & Dahl, A. B. (2023). *Fast Local Thickness*. IEEE/CVF Conference on Computer Vision and Pattern Recognition Workshops (CVPRW). https://doi.org/10.1109/cvprw59228.2023.00456